In [ ]:
import os
os.makedirs("../outputs_camelbert", exist_ok=True)

from transformers import AutoTokenizer, AutoModel

tokenizer_camelbert = AutoTokenizer.from_pretrained("CAMeL-Lab/bert-base-arabic-camelbert-da")
model_camelbert = AutoModel.from_pretrained("CAMeL-Lab/bert-base-arabic-camelbert-da")
model_camelbert.eval()

print("✅ CamelBERT (DA - dialectal) chargé !")

In [ ]:
exemple_avec_emoji = "صباحك جمال 🌿 🌹 🌿"
tokens_camelbert = tokenizer_camelbert.tokenize(exemple_avec_emoji)
print(f"Texte original : {exemple_avec_emoji}")
print(f"Tokens CamelBERT : {tokens_camelbert}")

encoded = tokenizer_camelbert(exemple_avec_emoji, return_tensors="pt")
print(f"\nInput IDs : {encoded['input_ids']}")
decoded = tokenizer_camelbert.decode(encoded['input_ids'][0])
print(f"Texte décodé : {decoded}")


In [ ]:
#Vérifier MAX_LENGTH

In [ ]:
longueurs_camelbert = df_train["cleaned_tweet"].apply(
    lambda t: len(tokenizer_camelbert.tokenize(t)) if isinstance(t, str) else 0
)
print(f"Longueur moyenne : {longueurs_camelbert.mean():.1f} tokens")
print(f"95e percentile   : {longueurs_camelbert.quantile(0.95):.1f} tokens")
print(f"99e percentile   : {longueurs_camelbert.quantile(0.99):.1f} tokens")

In [ ]:
longueurs_camelbert = df_train["cleaned_tweet"].apply(
    lambda t: len(tokenizer_camelbert.tokenize(t)) if isinstance(t, str) else 0
)
print(f"Longueur moyenne : {longueurs_camelbert.mean():.1f} tokens")
print(f"95e percentile   : {longueurs_camelbert.quantile(0.95):.1f} tokens")
print(f"99e percentile   : {longueurs_camelbert.quantile(0.99):.1f} tokens")

In [ ]:
import pandas as pd
import re
import pyarabic.araby as ar

df_train = pd.read_csv("../data/training_data.csv")
df_test = pd.read_csv("../data/testing_data.csv")

def clean_tweet(tweet):
    tweet = re.sub(r'http\S+|www.\S+', '', tweet)
    tweet = re.sub(r'@\w+|#\w+', '', tweet)
    tweet = re.sub(r'[^ء-ي\s]', '', tweet)
    tweet = ar.normalize_hamza(tweet)
    tweet = ar.normalize_ligature(tweet)
    tweet = ar.strip_tashkeel(tweet)
    return tweet.strip()

df_train["cleaned_tweet"] = df_train["tweet"].apply(clean_tweet)
df_test["cleaned_tweet"] = df_test["tweet"].apply(clean_tweet)

print("✅ Dataset ArSarcasm-v2 chargé et nettoyé !")
print(f"Train : {df_train.shape}")
print(f"Test  : {df_test.shape}")

In [ ]:
longueurs_camelbert = df_train["cleaned_tweet"].apply(
    lambda t: len(tokenizer_camelbert.tokenize(t)) if isinstance(t, str) else 0
)
print(f"Longueur moyenne : {longueurs_camelbert.mean():.1f} tokens")
print(f"95e percentile   : {longueurs_camelbert.quantile(0.95):.1f} tokens")
print(f"99e percentile   : {longueurs_camelbert.quantile(0.99):.1f} tokens")

In [ ]:
#l'extraction complète des embeddings CamelBERT sur ArSarcasm-v2

In [ ]:
import numpy as np
import torch
from tqdm import tqdm

os.makedirs("../outputs_camelbert", exist_ok=True)

MAX_LENGTH = 64

def get_embedding_sequence_camelbert(text):
    if not isinstance(text, str):
        text = ""
    inputs = tokenizer_camelbert(text, return_tensors="pt",
                                  truncation=True, max_length=MAX_LENGTH,
                                  padding="max_length")
    with torch.no_grad():
        outputs = model_camelbert(**inputs)
    sequence = outputs.last_hidden_state.squeeze(0).numpy()
    return sequence

N_train = len(df_train)
N_test = len(df_test)
seq_shape = (MAX_LENGTH, 768)

print(f"⏳ Extraction + sauvegarde directe train ({N_train} tweets)...")

train_mmap = np.lib.format.open_memmap(
    "../outputs_camelbert/train_embeddings_sequence.npy",
    mode='w+', dtype=np.float32, shape=(N_train,) + seq_shape
)

textes_train = df_train["cleaned_tweet"].tolist()
for i in tqdm(range(N_train)):
    train_mmap[i] = get_embedding_sequence_camelbert(textes_train[i])

train_mmap.flush()
del train_mmap

print("✅ Train embeddings sauvegardés !")

In [ ]:
print(f"⏳ Extraction + sauvegarde directe test ({N_test} tweets)...")

test_mmap = np.lib.format.open_memmap(
    "../outputs_camelbert/test_embeddings_sequence.npy",
    mode='w+', dtype=np.float32, shape=(N_test,) + seq_shape
)

textes_test = df_test["cleaned_tweet"].tolist()
for i in tqdm(range(N_test)):
    test_mmap[i] = get_embedding_sequence_camelbert(textes_test[i])

test_mmap.flush()
del test_mmap

print("✅ Test embeddings sauvegardés !")

np.save("../outputs_camelbert/train_labels.npy", df_train["sarcasm"].astype(int).values)
np.save("../outputs_camelbert/test_labels.npy", df_test["sarcasm"].astype(int).values)

print("✅ Tous les fichiers sauvegardés dans outputs_camelbert/ !")

In [ ]:
import gc
from sklearn.model_selection import train_test_split

gc.collect()

X_train_seq_camelbert = np.load("../outputs_camelbert/train_embeddings_sequence.npy", mmap_mode='r')
X_test_seq_camelbert = np.load("../outputs_camelbert/test_embeddings_sequence.npy", mmap_mode='r')

print("✅ Séquences CamelBERT rechargées en mode memmap !")
print(f"Train : {X_train_seq_camelbert.shape}")
print(f"Test  : {X_test_seq_camelbert.shape}")

train_prag_v2_full = pd.read_csv("../outputs/train_pragmatic_v2.csv")
test_prag_v2_full = pd.read_csv("../outputs/test_pragmatic_v2.csv")

X_train_prag_camelbert = train_prag_v2_full.drop("label", axis=1).values
X_test_prag_camelbert = test_prag_v2_full.drop("label", axis=1).values

y_train_full_camelbert = np.load("../outputs_camelbert/train_labels.npy")
y_test_camelbert = np.load("../outputs_camelbert/test_labels.npy")

indices_camelbert = np.arange(len(y_train_full_camelbert))
idx_train_camelbert, idx_val_camelbert = train_test_split(
    indices_camelbert, test_size=0.15, stratify=y_train_full_camelbert, random_state=42
)

print(f"\n✅ Split train/val créé !")
print(f"Train final : {len(idx_train_camelbert)} | Val : {len(idx_val_camelbert)}")
print(f"Test : {len(y_test_camelbert)}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import copy
from sklearn.metrics import f1_score, matthews_corrcoef, classification_report, precision_score, recall_score, accuracy_score
from scipy.stats import wilcoxon

class SoftAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(SoftAttention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)
    def forward(self, lstm_output):
        scores = self.attention(lstm_output).squeeze(-1)
        weights = F.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)
        return context, weights

class SAttBLSTMConvNet(nn.Module):
    def __init__(self, embed_dim=768, conv_channels=128, lstm_hidden=128, prag_dim=18, dropout=0.3):
        super(SAttBLSTMConvNet, self).__init__()
        self.conv1 = nn.Conv1d(embed_dim, conv_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.dropout_conv = nn.Dropout(dropout)
        self.bilstm = nn.LSTM(conv_channels, lstm_hidden, batch_first=True, bidirectional=True)
        self.attention = SoftAttention(lstm_hidden * 2)
        self.dropout_final = nn.Dropout(dropout)
        self.fc1 = nn.Linear(lstm_hidden * 2 + prag_dim, 128)
        self.fc2 = nn.Linear(128, 2)
    def forward(self, x_seq, x_prag):
        x = x_seq.permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)
        lstm_out, _ = self.bilstm(x)
        context, attn_weights = self.attention(lstm_out)
        combined = torch.cat([context, x_prag], dim=1)
        combined = self.dropout_final(combined)
        out = self.relu(self.fc1(combined))
        return self.fc2(out)

class SarcasmDatasetMemmap(Dataset):
    def __init__(self, X_seq_memmap, X_prag, y, indices):
        self.X_seq_memmap = X_seq_memmap
        self.X_prag = torch.FloatTensor(X_prag[indices])
        self.y = torch.LongTensor(y[indices])
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        seq = np.array(self.X_seq_memmap[self.indices[idx]])
        return torch.FloatTensor(seq), self.X_prag[idx], self.y[idx]

def entrainer_avec_early_stopping_memmap(X_seq_memmap, X_prag, y, idx_train, idx_val,
                                           lr=0.001, max_epochs=50, patience=10, batch_size=32):
    train_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_val)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    class_counts = np.bincount(y[idx_train])
    weights = torch.FloatTensor(1.0 / class_counts)
    model = SAttBLSTMConvNet(embed_dim=768, prag_dim=X_prag.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=weights)
    best_f1 = 0
    best_model_state = None
    epochs_sans_amelioration = 0
    for epoch in range(max_epochs):
        model.train()
        total_loss = 0
        for x_seq_b, x_prag_b, y_b in train_loader:
            optimizer.zero_grad()
            outputs = model(x_seq_b, x_prag_b)
            loss = criterion(outputs, y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for x_seq_b, x_prag_b, y_b in val_loader:
                outputs = model(x_seq_b, x_prag_b)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.numpy())
                all_labels.extend(y_b.numpy())
        f1_val = f1_score(all_labels, all_preds, average='macro')
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"   Epoch {epoch+1}/{max_epochs} - Loss: {total_loss/len(train_loader):.4f} - F1 val: {f1_val:.4f}")
        if f1_val > best_f1:
            best_f1 = f1_val
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_sans_amelioration = 0
        else:
            epochs_sans_amelioration += 1
        if epochs_sans_amelioration >= patience:
            print(f"   ⏹️ Early stopping à l'époque {epoch+1}")
            break
    model.load_state_dict(best_model_state)
    print(f"✅ Meilleur F1 validation atteint : {best_f1:.4f}")
    return model

def evaluer_test_complet_memmap(model, X_seq_memmap, X_prag, y, idx_test, batch_size=32):
    test_dataset = SarcasmDatasetMemmap(X_seq_memmap, X_prag, y, idx_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_seq_b, x_prag_b, y_b in test_loader:
            outputs = model(x_seq_b, x_prag_b)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(y_b.numpy())
    resultats = {
        "F1_macro": f1_score(all_labels, all_preds, average='macro'),
        "Precision_macro": precision_score(all_labels, all_preds, average='macro'),
        "Recall_macro": recall_score(all_labels, all_preds, average='macro'),
        "Accuracy": accuracy_score(all_labels, all_preds),
        "MCC": matthews_corrcoef(all_labels, all_preds),
        "F1_sarcastique": f1_score(all_labels, all_preds, pos_label=1),
        "Precision_sarcastique": precision_score(all_labels, all_preds, pos_label=1),
        "Recall_sarcastique": recall_score(all_labels, all_preds, pos_label=1)
    }
    print(classification_report(all_labels, all_preds))
    print(f"\n{'='*50}\nRÉSUMÉ DES MÉTRIQUES\n{'='*50}")
    for nom, valeur in resultats.items():
        print(f"{nom:25} : {valeur:.4f}")
    return resultats

print("✅ Architecture et fonctions prêtes !")

In [ ]:
def evaluer_hyperparam_camelbert(lr, epochs=5, batch_size=32):
    torch.manual_seed(42)
    
    train_dataset = SarcasmDatasetMemmap(X_train_seq_camelbert, X_train_prag_camelbert, y_train_full_camelbert, idx_train_camelbert)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    val_dataset = SarcasmDatasetMemmap(X_train_seq_camelbert, X_train_prag_camelbert, y_train_full_camelbert, idx_val_camelbert)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    class_counts = np.bincount(y_train_full_camelbert[idx_train_camelbert])
    weights = torch.FloatTensor(1.0 / class_counts)
    
    model = SAttBLSTMConvNet(embed_dim=768, prag_dim=18)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=weights)
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for x_seq, x_prag, y in train_loader:
            optimizer.zero_grad()
            outputs = model(x_seq, x_prag)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_seq, x_prag, y in val_loader:
            outputs = model(x_seq, x_prag)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(y.numpy())
    
    f1_val = f1_score(all_labels, all_preds, average='macro')
    return f1_val, total_loss / len(train_loader)


print("⏳ Test de plusieurs learning rates (ArSarcasm-v2, CamelBERT)...\n")
learning_rates = [0.001, 0.0001, 0.00001]
resultats_hyperparam_camelbert = []

for lr in learning_rates:
    print(f"⏳ Test avec lr={lr}...")
    f1_val, loss_final = evaluer_hyperparam_camelbert(lr, epochs=5)
    print(f"   F1 macro (validation) = {f1_val:.4f} | Loss finale = {loss_final:.4f}\n")
    resultats_hyperparam_camelbert.append({"learning_rate": lr, "F1_validation": f1_val, "loss": loss_final})

hyperparam_camelbert_df = pd.DataFrame(resultats_hyperparam_camelbert)
hyperparam_camelbert_df = hyperparam_camelbert_df.sort_values("F1_validation", ascending=False)

print("="*50)
print("RÉSULTATS — Comparaison learning rates (CamelBERT)")
print("="*50)
print(hyperparam_camelbert_df.to_string(index=False))

In [ ]:
#les 5 runs SANS features

In [ ]:
seeds = [42, 123, 456, 789, 101]

resultats_5runs_sans_camelbert = []

X_train_prag_zeros_camelbert = np.zeros_like(X_train_prag_camelbert)
X_test_prag_zeros_camelbert = np.zeros_like(X_test_prag_camelbert)

print("="*60)
print("5 RUNS — SANS features pragmatiques (ArSarcasm-v2, CamelBERT)")
print("="*60)

for i, seed in enumerate(seeds):
    print(f"\n--- Run {i+1}/5 (seed={seed}) ---")
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    model_s = entrainer_avec_early_stopping_memmap(
        X_train_seq_camelbert, X_train_prag_zeros_camelbert, y_train_full_camelbert,
        idx_train_camelbert, idx_val_camelbert,
        lr=0.001, max_epochs=50, patience=10
    )
    
    idx_test_all_camelbert = np.arange(len(y_test_camelbert))
    resultats_metriques = evaluer_test_complet_memmap(
        model_s, X_test_seq_camelbert, X_test_prag_zeros_camelbert, y_test_camelbert, idx_test_all_camelbert
    )
    resultats_metriques["seed"] = seed
    resultats_5runs_sans_camelbert.append(resultats_metriques)

print("\n✅ 5 runs SANS features terminés !")
pd.DataFrame(resultats_5runs_sans_camelbert).to_csv("../outputs_camelbert/5runs_sans_features_complet.csv", index=False)
print("✅ Résultats sauvegardés !")

In [ ]:
#es 5 runs AVEC features pragmatiques

In [ ]:
resultats_5runs_avec_camelbert = []

print("="*60)
print("5 RUNS — AVEC features pragmatiques (ArSarcasm-v2, CamelBERT)")
print("="*60)

for i, seed in enumerate(seeds):
    print(f"\n--- Run {i+1}/5 (seed={seed}) ---")
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    model_a = entrainer_avec_early_stopping_memmap(
        X_train_seq_camelbert, X_train_prag_camelbert, y_train_full_camelbert,
        idx_train_camelbert, idx_val_camelbert,
        lr=0.001, max_epochs=50, patience=10
    )
    
    idx_test_all_camelbert = np.arange(len(y_test_camelbert))
    resultats_metriques = evaluer_test_complet_memmap(
        model_a, X_test_seq_camelbert, X_test_prag_camelbert, y_test_camelbert, idx_test_all_camelbert
    )
    resultats_metriques["seed"] = seed
    resultats_5runs_avec_camelbert.append(resultats_metriques)

print("\n✅ 5 runs AVEC features terminés !")
pd.DataFrame(resultats_5runs_avec_camelbert).to_csv("../outputs_camelbert/5runs_avec_features_complet.csv", index=False)
print("✅ Résultats sauvegardés !")

In [ ]:
#Test de Wilcoxon

In [ ]:
f1_sans_camelbert = [r["F1_macro"] for r in resultats_5runs_sans_camelbert]
f1_avec_camelbert = [r["F1_macro"] for r in resultats_5runs_avec_camelbert]

print("F1 SANS :", [f"{f:.4f}" for f in f1_sans_camelbert])
print("F1 AVEC :", [f"{f:.4f}" for f in f1_avec_camelbert])

print(f"\nSANS : {np.mean(f1_sans_camelbert):.4f} ± {np.std(f1_sans_camelbert):.4f}")
print(f"AVEC : {np.mean(f1_avec_camelbert):.4f} ± {np.std(f1_avec_camelbert):.4f}")

stat, p_value = wilcoxon(f1_avec_camelbert, f1_sans_camelbert, alternative='greater')
print(f"\nWilcoxon (ArSarcasm-v2, CamelBERT) : p = {p_value:.4f}")

if p_value < 0.05:
    print("✅ SIGNIFICATIF")
elif p_value < 0.10:
    print("⚠️ MARGINALEMENT significatif")
else:
    print("⚠️ NON significatif")